# Lesson 57 · From one neuron to a whole network — an MLP from scratch, with all 41 parameters lit up by you

**Goal**: implement `Neuron → Layer → MLP`, with every level supporting forward / backward.

> **The story in layers**: 1 neuron → one layer → the whole network; the parameter table follows the code. The weights **must not all be 0** (the symmetry trap).

🔗 **Aurora connection**: this MLP has exactly the same structure as PyTorch's `nn.Module`, and it is the prototype for the Aurora ML engine (`src/aurora/ml/`, a planned module). In **L61** `L61_pytorch_nn.ipynb` we rebuild the same three-layer structure in PyTorch; you'll then be able to compare the two pieces of code line by line and confirm that your implementation behaves the same as the framework.

← **Previous lesson** · [L56 · Backpropagation by hand](L56_backward_pass.ipynb)

> Last lesson covered **backpropagation by hand**: the chain rule unrolled layer by layer, gradient = local gradient × upstream gradient.
> This lesson actually builds the **Neuron → Layer → MLP** set of building blocks.

## This lesson's story: from a single neuron to a network that "talks"

Ever wondered why the billions of parameters inside GPT can all be updated by the same single line of code, `loss.backward()`?

The answer lies in **how it is assembled**. A neural network is not one big black box; it is three levels of nesting:

```
Neuron  ← the smallest unit: one weighted sum + an activation function
Layer   ← the middle level: nout Neurons side by side, all receiving the same input
MLP     ← the complete network: several Layers chained end to end
```

**What one Neuron does** (nin=2, outputs one scalar):
```
output = relu( w[0]*x[0] + w[1]*x[1] + b )
       = relu( Value(0.7)*Value(1.0) + Value(-0.3)*Value(2.0) + Value(0.1) )
       = relu( Value(0.7 - 0.6 + 0.1) )
       = relu( Value(0.2) ) = Value(0.2)
```

Every step is a `Value` operator you implemented in the previous lessons: the computation graph builds itself, and `backward()` passes the gradients back automatically.

**Why it is designed this way**:

| Level | Role | Number of parameters |
|---|---|---|
| Neuron(nin) | 1 linear weighting + activation | nin+1 |
| Layer(nin, nout) | nout Neurons | nout×(nin+1) |
| MLP(nin, [n1,n2,...]) | several layers in series | summed layer by layer |

Your task in this lesson: implement three classes so that `MLP(3,[4,4,1])` can run forward, can run `backward()`, and can count its parameters.

In [ ]:
import random

# ── The Value class (the full implementation from L54–L56, pasted here so this lesson runs on its own) ──
import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)

    def __truediv__(self, other):
        # Must use other**-1 (which calls __pow__), not Value(other.data**-1).
        # Value(x**-1) creates an isolated node: it isn't connected to the original other, so the gradient can't flow back.
        # other**-1 builds a 1/other node with other as its child, so the gradient chain is complete.
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# Quick smoke test
a = Value(2.0); b = Value(3.0)
c = (a * b + Value(1.0)).tanh()
c.backward()
assert abs(a.grad) > 0, "Value.backward() is not working"
print("✅ Value class loaded")

# Division gradient check (node connectivity check)
x = Value(6.0)
y = Value(2.0)
z = x / y          # z = 3.0
z.backward()
# d(x/y)/dx = 1/y = 0.5,  d(x/y)/dy = -x/y^2 = -1.5
assert abs(x.grad - 0.5) < 1e-9,   f"dz/dx is wrong: {x.grad} (should be 0.5)"
assert abs(y.grad - (-1.5)) < 1e-9, f"dz/dy is wrong: {y.grad} (should be -1.5)"
print(f"✅ Division gradients correct: dz/dx={x.grad:.4f}, dz/dy={y.grad:.4f}")

## 1. Neuron — a single neuron

**What to implement**:

```python
class Neuron(nin, nonlin=True):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(0.0)

    def __call__(x):  # x: list[Value or float]
        act = sum(wi * xi for wi, xi in zip(self.w, x)) + self.b
        return act.tanh() if self.nonlin else act

    def parameters() → list[Value]  # returns w + [b]
```

(The original micrograd uses `relu`; this lesson uses `tanh` throughout as the hidden-layer activation — its output lands in (-1, 1), which makes it easy to watch. Both are operators you already implemented in L55, so feel free to swap one for the other and compare.)

**Acceptance criteria**:
- `Neuron(3)(x)` returns a single `Value`, and its `.data` is finite
- `len(Neuron(3).parameters()) == 4` (3 w's + 1 b)
- with `nonlin=False` it returns the linear combination directly, without tanh

### Counting parameters: why does Layer(nin, nout) have nout×(nin+1) parameters?

It looks like just a formula, but understanding the logic behind it lets you quickly check the parameter total of any network.

**Start from one Neuron**:
- `Neuron(nin)` needs `nin` weights (w[0], w[1], ..., w[nin-1]) plus 1 bias (b)
- that's **nin + 1** parameters in total

**Move up to a Layer**:
- `Layer(nin, nout)` contains `nout` independent `Neuron(nin)`s
- Neuron 1 has nin+1 parameters
- Neuron 2 has nin+1 parameters
- ...
- Neuron nout has nin+1 parameters

**Add them up**:
```
Layer's total parameters = nout Neurons × (nin+1) parameters each
                         = nout × (nin + 1)
                         = nout×nin + nout×1
                         = (number of weights) + (number of biases)
```

**A concrete example**: `Layer(3, 4)` has 4 Neurons, each handling 3 inputs
- weights = 3 × 4 = 12 (3 weights per Neuron, 4 Neurons)
- biases = 4 (1 bias per Neuron)
- **total = 16**

That's why the table we see in the code looks the way it does.

### Warm-up: understanding `zip()` "pairing"

In the Neuron's forward computation we need to multiply the weight list `w` and the input list `x` element by element: `w[0]*x[0], w[1]*x[1], ...`.

Python's `zip()` exists for exactly this kind of "pairing":

```python
w = [0.5, -0.3, 0.2]      # 3 weights
x = [1.0,  2.0, 3.0]      # 3 inputs

# zip(w, x) pairs up the two lists
for wi, xi in zip(w, x):
    print(f"w={wi}, x={xi}")
# Output:
# w=0.5, x=1.0
# w=-0.3, x=2.0
# w=0.2, x=3.0
```

This lets you compute the product `wi * xi` as you iterate. If you've only ever written `for i in range(n)`, this "direct unpacking" may feel unfamiliar, but it's very handy — one glance at the code tells you that you're pairing up two sequences.

In [ ]:
# A minimal check of zip()
w_test = [0.5, -0.3, 0.2]
x_test = [1.0, 2.0, 3.0]
pairs = list(zip(w_test, x_test))
print("zip output (as a list):", pairs)
print("products, pair by pair:", [wi*xi for wi, xi in zip(w_test, x_test)])

### Warm-up: the smallest Value example — where the computation graph comes from

When you're starting out, the phrase "Value objects build a computation graph" can feel abstract. Let's look at the simplest possible example:

```python
a = Value(2.0)   # create a Value that stores the data 2.0
b = Value(3.0)   # create another Value that stores the data 3.0
c = a + b        # compute c = 2.0 + 3.0 = 5.0
# c is now a new Value that records "I was produced from a and b by addition"
```

When we call `c.backward()`, the system will:
1. start from `c` (setting `c.grad = 1.0`)
2. walk backwards along the "who produced me" chain: `c` was produced by `a+b`
3. compute the local gradients: `dc/da = 1`, `dc/db = 1`
4. end up with values filled in for both `a.grad` and `b.grad`

**The key point**: when we define `c = a + b`, the source code of `Value.__add__` records "c was produced from a and b" — that is the **computation graph**. Let's check it:

In [ ]:
# Minimal example: 1 + 1 = 2, then backward()
a = Value(1.0)
b = Value(1.0)
c = a + b        # c.data = 2.0, c._prev records {a, b}
print(f"c.data = {c.data}  (should be 2.0)")
print(f"children of c: {c._prev}  (should contain a and b)")

# backward() walks the computation graph backwards, updating gradients
c.backward()
print(f"after backprop: a.grad = {a.grad}, b.grad = {b.grad}  (both should be 1.0)")
print("✅ The Value computation graph works")

### Illustrated: why can't `__truediv__` be written as `Value(other.data ** -1)`?

The Value class's `__truediv__` above carries this comment:

```python
def __truediv__(self, other):
    # Must use other**-1 (which calls __pow__), not Value(other.data**-1).
    # Value(x**-1) creates an isolated node: it isn't connected to the original other, so the gradient can't flow back.
    # other**-1 builds a 1/other node with other as its child, so the gradient chain is complete.
    ...
```

The words say "an isolated node can't pass the gradient back", which may still feel abstract. Building on what we just saw with `c._prev`, let's draw the computation graphs of the two versions side by side.

**First, an analogy**: a computation graph is like a "family tree" — every Value, when it is born, has to register "who my parents are" in `_prev`. `backward()` walks this family tree from child to parent, passing the gradient all the way back. If a node's `_prev` is empty (no parents registered), it is an "orphan" outside the tree — the gradient stops there and goes no further.

**❌ Wrong version: `Value(other.data ** -1)`**

```
   other (data=4.0)                 ← this is the object that really takes part in the later computation
                                        but it is never "registered" as anyone's parent!

   Value(other.data ** -1)  =  Value(0.25)     ← a brand-new object made out of thin air
        _prev = {}                              (empty set, no parent nodes at all)
        data  = 0.25   (numerically equal to 1/other, but it only "copied the number" over)

   result = self * Value(0.25)
        _prev = {self, Value(0.25)}

   backward() starts from result and can reach self, and can reach Value(0.25),
   but Value(0.25)._prev is empty — the road ends here.
   The real object, other, never appears on any _prev edge from start to finish,
   so other.grad stays stuck at its initial value 0 — the gradient "can't get back".
```

**✅ Right version: `other ** -1`**

```
   other (data=4.0) ──(**-1, calls __pow__)──→ pow_node (data=0.25)
                                                  _prev = {other}   ← registered!

   result = self * pow_node
        _prev = {self, pow_node}

   backward() starts from result → reaches pow_node (because pow_node ∈ result._prev)
                                 → then reaches other (because other ∈ pow_node._prev)
   One complete path: the gradient travels back along it step by step to other,
   and other.grad correctly accumulates a nonzero value.
```

**In one sentence**: `other ** -1` and `Value(other.data ** -1)` compute exactly the same **number** (both 0.25); the only difference is that the former "remembers it was computed from other" and the latter doesn't. Backpropagation doesn't run on the numbers but on this record of "who produced whom" — get the number right but lose the record, and the gradient still can't get back.

The code below runs both versions so you can see the difference in `other.grad` for yourself:

In [ ]:
# See for yourself: an "isolated node" vs a "connected node" in the computation graph

# ❌ Wrong version: compute 1/other by hand and wrap it in a brand-new Value
other_bad = Value(4.0)
isolated = Value(other_bad.data ** -1)     # data=0.25, but _prev is empty
result_bad = Value(2.0) * isolated
result_bad.backward()

print("❌ Wrong version Value(other.data ** -1):")
print(f"   isolated._prev = {isolated._prev}   (empty set — an orphan node)")
print(f"   other_bad.grad = {other_bad.grad}   (stuck at 0.0, the gradient can't get back!)")

print()

# ✅ Right version: other ** -1, pow_node remembers that other is its parent
other_good = Value(4.0)
pow_node = other_good ** -1                # data=0.25, _prev={other_good}
result_good = Value(2.0) * pow_node
result_good.backward()

print("✅ Right version other ** -1:")
print(f"   pow_node._prev = {pow_node._prev}   (contains other_good itself)")
print(f"   other_good.grad = {other_good.grad}   (a nonzero gradient, computed correctly)")

In [ ]:
# Demo: build an nin=2 Neuron by hand (no class yet — just write out the operations)
random.seed(42)
w = [Value(random.uniform(-1, 1)) for _ in range(2)]
b  = Value(0.0)

x  = [Value(1.5), Value(-0.5)]

# Forward: dot + bias + tanh
act = sum((wi * xi for wi, xi in zip(w, x)), Value(0.0)) + b
out = act.tanh()

print("w  =", [round(wi.data, 4) for wi in w])
print("b  =", round(b.data, 4))
print("x  =", [xi.data for xi in x])
print("act=", round(act.data, 4))
print("out=", round(out.data, 4), "  (tanh values lie between -1 and 1)")

### Extra: how backward() passes gradients — the chain rule on autopilot

What does the system actually do when you call `loss.backward()`?

**Initialization**:
```python
loss.grad = 1.0   # start from the output: set dloss/dloss = 1.0
```

**Walk the computation graph in reverse order**:
In `__init__`, the Value class records "who produced me" (in the `_prev` set). The core of backward() is:

```python
# pseudocode
for node in reverse_topological_order(graph):
    node._backward()   # call this node's local gradient function
```

**What is a local gradient?**

Every operation (+, ×, tanh, …) has its own gradient rule. For example:

```python
# Addition: c = a + b
# dc/da = 1, dc/db = 1
def add_backward():
    a.grad += c.grad * 1      # local gradient of c with respect to a = 1
    b.grad += c.grad * 1      # local gradient of c with respect to b = 1

# Multiplication: c = a * b
# dc/da = b, dc/db = a
def mul_backward():
    a.grad += c.grad * b.data    # local gradient of c with respect to a = b
    b.grad += c.grad * a.data    # local gradient of c with respect to b = a

# tanh: y = tanh(z)
# dy/dz = 1 - tanh(z)^2
def tanh_backward():
    z.grad += y.grad * (1 - y.data**2)
```

**The chain rule on autopilot**:

When you see code like `a.grad += c.grad * local_gradient`, what it is really computing is:

```
dloss/da = dloss/dc × dc/da
           └upstream┘  └local┘
```

If a → c → loss, then:

```
dloss/da = dloss/dc × dc/da
         = c.grad (already computed in the previous step) × local_gradient
```

**A concrete example**

```python
a = Value(2.0)
b = Value(3.0)
c = a + b           # c.grad gets filled in during loss.backward()
loss = c * Value(2.0)

loss.backward()
# Step 1: loss's backward (multiplication: loss = c × 2.0)
#   c.grad += loss.grad * 2.0  = 1.0 * 2.0 = 2.0
#   (the constant Value(2.0) gets grad += loss.grad * c.data = 5.0)

# Step 2: c's backward (addition, passing c.grad further up)
#   a.grad += c.grad * 1  = 0.0 + 2.0 * 1 = 2.0
#   b.grad += c.grad * 1  = 0.0 + 2.0 * 1 = 2.0

print(f"a.grad = {a.grad}, b.grad = {b.grad}")
# a.grad = 2.0 (passed through two layers: × then +)
# b.grad = 2.0 (same path as a — addition hands both inputs the same gradient)
```

In [ ]:
# A live demo of gradient propagation
a = Value(2.0)
b = Value(3.0)
c = a + b           # c = 5.0
loss = c * Value(2.0)  # loss = 10.0

print(f"forward pass: a={a.data}, b={b.data}, c={c.data}, loss={loss.data}")
print(f"initial gradients: a.grad={a.grad}, b.grad={b.grad}, c.grad={c.grad}, loss.grad={loss.grad}")

loss.backward()

print(f"\nafter backward():")
print(f"  a.grad = {a.grad}  (should be 2.0, because dloss/da = dloss/dc × dc/da = 2.0 × 1.0)")
print(f"  b.grad = {b.grad}  (should be 2.0, because dloss/db = dloss/dc × dc/db = 2.0 × 1.0)")
print(f"  c.grad = {c.grad}  (should be 2.0, because dloss/dc = 2.0 (from the multiplication))")

# Now a more involved example: a affects loss along two paths
print("\n" + "="*50)
print("A more involved example: a reaches loss along several paths")
a2 = Value(2.0)
c2 = a2 * a2        # c = a * a = 4
d2 = c2 + a2        # d = c + a = 4 + 2 = 6
loss2 = d2 * Value(3.0)  # loss = d * 3 = 18

print(f"forward: a2={a2.data}, c2={c2.data}, d2={d2.data}, loss2={loss2.data}")
loss2.backward()

print(f"after backward(): a2.grad = {a2.grad}")
print(f"  expected: dloss/da = 3×(dc/da + dd/da)")
print(f"              = 3×(2×a + 1)")
print(f"              = 3×(2×2 + 1)")
print(f"              = 3×5 = 15")
print(f"  actual value: {a2.grad}  ✅")


### Extra: how `sum()` works on Value objects

In the Neuron's computation you'll see this line:

```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
```

It looks complicated, but the logic is simple — let's take it apart step by step:

```python
# Step 1: produce all the products (a generator expression)
products = (wi * xi for wi, xi in zip(self.w, x))  # not run yet, only defined

# Step 2: sum() adds them up one by one
result = sum(products, Value(0.0))
# equivalent to:
# result = Value(0.0)
# for product in products:
#     result = result + product
```

**Why is the second argument `Value(0.0)`?**

In Python's `sum()` function, the second argument is the **start value**. For example:

```python
sum([1, 2, 3], 0)      # 0 + 1 + 2 + 3 = 6
sum([1, 2, 3], 10)     # 10 + 1 + 2 + 3 = 16
```

It works the same way for Value objects:

```python
sum([v1, v2, v3], Value(0.0))   # start from Value(0.0) and add one at a time
                                # = Value(0.0) + v1 + v2 + v3
```

**What happens with plain `sum(...)` (no second argument)?**

If you write `sum((wi * xi for ...), 0)` (with an ordinary 0 — which is also the default when you leave the second argument out), the first step is:

```
0 + (w[0]*x[0])   # here 0 is an int and (w[0]*x[0]) is a Value
```

Python's `int.__add__` has no idea how to add a Value object. On its own that would be an error; it only works because our `Value` class defines `__radd__`, which Python falls back to. Making the start value **a Value object** removes that dependency: `Value.__add__` takes over the addition from the very first step, and the computation graph gets built.

Let's look at an example:

In [ ]:
# An example of sum() with Value objects
v_list = [Value(2.0), Value(3.0), Value(5.0)]

# ✅ The safe way: use Value(0.0) as the start value
result_correct = sum(v_list, Value(0.0))
print(f"sum(Value list, Value(0.0)) = {result_correct.data}  (should be 10.0)")

# ⚠️ Plain 0 as the start value: works here only because Value defines __radd__
try:
    result_wrong = sum(v_list, 0)
    print(f"sum(Value list, 0) = {result_wrong}")  # int.__add__ gives up and Value.__radd__ steps in; without __radd__, 0 + Value(2.0) would fail here
except TypeError as e:
    print(f"❌ sum(Value list, 0) failed: {type(e).__name__}: int doesn't know how to add a Value")

# Check the computation graph
result_correct.backward()
for i, v in enumerate(v_list):
    print(f"v_list[{i}].grad = {v.grad}  (all should be 1.0, meaning gradients propagate correctly)")

### Extra: tanh vs relu — why does this lesson pick tanh?

You may have noticed the note saying "the original micrograd uses relu; this lesson uses tanh". What's the difference between them?

| Activation | Output range | Formula | When to use it |
|---------|--------|------|----------|
| **tanh** | (-1, 1) | tanh(z) = (e^z - e^{-z}) / (e^z + e^{-z}) | this lesson's hidden layers |
| **relu** | [0, ∞) | relu(z) = max(0, z) | modern deep learning (faster, sparser) |

**Why does this lesson pick tanh?**

1. **Nice symmetry**: the output is centered on 0 (-1 to 1), so gradients are more stable
2. **Easy to watch**: outputs all lie in (-1, 1) — small numbers that are easy to print and debug
3. **Simple gradient**: d(tanh)/dz = 1 - tanh(z)^2 (smooth and easy to understand)

**Where relu wins**:
- **Fast to compute**: relu is just `max(0, z)`, with no exponentials
- **Sparse activation**: negatives go straight to 0, so the network learns on its own which features don't matter
- **Friendly to deep networks**: its gradient doesn't "saturate" the way tanh's does (when |z| is large, tanh's gradient is close to 0)

**You already have a Value.relu() method** (you can see it in the code), so if you want to try relu, just change `.tanh()` to `.relu()` in the code.

Let's compare the gradients quickly:

In [ ]:
import math
import numpy as np

# Compare the outputs and gradients of tanh and relu
z_vals = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

print("z value    | tanh(z)    | d(tanh)/dz | relu(z)  | d(relu)/dz")
print("-" * 65)
for z in z_vals:
    tanh_z = math.tanh(z)
    d_tanh = 1 - tanh_z**2
    relu_z = max(0, z)
    d_relu = 1.0 if z > 0 else 0.0
    print(f"{z:6.1f}     | {tanh_z:8.4f}   | {d_tanh:8.4f}  | {relu_z:7.1f} | {d_relu:8.1f}")

print("\nWhat to notice:")
print("- tanh's gradient is always positive, in (0, 1], and close to 0 when |z| is large (vanishing gradient)")
print("- relu's gradient is either 0 or 1 (binary): fast to compute, fully off for negatives, fully on for positives")

## 2. Layer — a group of Neurons side by side

`Layer(nin, nout)` contains `nout` independent `Neuron(nin)`s. They all receive **the same input vector `x`**, each computes on its own, and the output is a list of `nout` scalars:

```
[neuron_0(x), neuron_1(x), ..., neuron_{nout-1}(x)]
```

With `nout=1` the output list has length 1 (a common setting for the final regression layer). Each `Neuron`'s parameters are independent of the others, and their gradients don't interfere.

### A closer look: Layer's "smart return" — why does a single output come back as a Value?

In the code comments for Layer.__call__ you'll see a design choice:

```python
if len(outs) == 1:
    return outs[0]      # a single output comes back as a Value (scalar)
else:
    return outs         # several outputs come back as a list
```

It looks convenient, but why design it this way?

**The heart of the problem**: how do the layers of a multi-layer MLP chain together?

Imagine you have three Layers:
- Layer1(3, 4) outputs 4 values
- Layer2(4, 2) should take 4 values as input
- Layer3(2, 1) should take 2 values as input, and **outputs only 1**

**Two possible designs**:

Design A (always return a list):
```python
out1 = layer1(x)          # returns [v0, v1, v2, v3]
out2 = layer2(out1)       # returns [v4, v5]
out3 = layer3(out2)       # returns [v6]  <- only 1 value, but still a list
loss = out3[0]            # you still need [0] to get it out
```
Every time you use the loss you have to write `out3[0]` — annoying.

Design B (smart return, the one this lesson uses):
```python
out1 = layer1(x)          # returns [v0, v1, v2, v3]
out2 = layer2(out1)       # returns [v4, v5]
out3 = layer3(out2)       # automatically returns v6 (a single Value, not a list)
loss = out3               # use it directly, no [0] needed
```

**The cost**: `MLP.__call__` has to "intercept" the result, making sure that even when the last layer is a single value, it is wrapped into a list before being handed to the outside world (the training loop in the next lesson will need this).

This is **interface design**: give up a little consistency at the right level in exchange for convenience in use.

In [ ]:
# Demo: build an nout=3 Layer by hand (nin=2 → outputs 3 scalars)
random.seed(0)
nin, nout = 2, 3
# 3 Neurons, each with 2 weights
neurons_demo = [
    [Value(random.uniform(-1, 1)) for _ in range(nin)]
    for _ in range(nout)
]
biases_demo = [Value(0.0) for _ in range(nout)]

x_demo = [Value(1.0), Value(-1.0)]

outputs_demo = []
for ws, b in zip(neurons_demo, biases_demo):
    act = sum((w * xi for w, xi in zip(ws, x_demo)), Value(0.0)) + b
    outputs_demo.append(act.tanh())

print(f"Layer(nin=2, nout=3) outputs {len(outputs_demo)} Values:")
for i, o in enumerate(outputs_demo):
    print(f"  out[{i}] = {o.data:.4f}")

### A closer look: if a middle layer has nout=1, does the "smart return" cause trouble?

There's a sharp question worth digging into: **a Layer sometimes returns a Value and sometimes returns a list — so when two layers are chained end to end, can the next layer actually catch whatever the previous one "casually" hands it?**

First, look at what an MLP usually looks like: `MLP(3, [4, 4, 1])` — the first two layers both have `nout` = 4 (≥2), and only **the last layer** has `nout=1`. That's no accident. Let's see what happens if a layer with `nout=1` shows up **in the middle**.

**Recall the Neuron's forward code**:
```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
```
Here `x` must be **iterable** (a list, say), because `zip(self.w, x)` has to pair up `self.w` and `x` element by element. If `x` is a bare `Value` object (not a list), `zip()` has no idea how to "pair" it — it raises an error straight away, because `Value` doesn't implement `__iter__` and can't be iterated.

**How exactly does it go wrong?** Run it once with a "bottleneck layer" (a middle layer with nout=1) and you'll see:

In [ ]:
# Demo: with a middle layer of nout=1, does the "smart return" trip up?
# (two mini demo classes written here; they don't affect the Neuron/Layer you implement later)

class _DemoNeuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
        return act.tanh()

class _DemoLayer:
    def __init__(self, nin, nout):
        self.neurons = [_DemoNeuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs   # this is the "smart return"

random.seed(7)
layer_a = _DemoLayer(3, 1)   # a middle layer, nout=1 (a bottleneck) — not the last layer of the network!
layer_b = _DemoLayer(1, 2)   # the next layer, nin=1, expects "a list with 1 element"

x_in = [Value(1.0), Value(2.0), Value(3.0)]
out_a = layer_a(x_in)
print(f"output type of layer_a(nout=1): {type(out_a).__name__}")
print("  ← because of the smart-return rule, nout=1 returns a bare Value, not [Value]!")

try:
    out_b = layer_b(out_a)   # the Neurons inside layer_b will try zip(self.w, out_a)
    print("layer_b output:", out_b)
except TypeError as e:
    print(f"\n❌ Error! TypeError: {e}")
    print("Reason: out_a is a bare Value, which is not iterable;")
    print("        zip(self.w, out_a) can't pair the elements of self.w with out_a one by one.")

**What does this tell us?**

The "smart return" design has a cost — it assumes that **only the last layer of the network** hands its result straight to an outside user (a human or the loss function). A middle layer's output is always just internal data "fed to the next layer", and it **should never** have nout=1; otherwise the next layer's `zip()` fails straight away, exactly as above.

**How do this course's examples avoid the trap?** Look at `MLP(3, [4, 4, 1])` — the width sequence is `[4, 4, 1]`, and only **the last number** is 1; the hidden-layer widths before it are all ≥ 2. This is a **convention**, not something the code guarantees: if you design your own architecture and write something like `MLP(3, [4, 1, 4])` (with a bottleneck layer hidden in the middle), the forward pass will fail at the joint between that middle layer and the next one, just as the demo showed.

**There is only one place inside the MLP that really smooths out the type difference**: at the very end, `MLP.__call__` wraps **the whole network's final output** into a list before returning it to the outside caller (a later subsection of this lesson goes into this in detail). The hand-off between layers (the `x = layer(x)` step) has no extra protection — this is a known limitation of this from-scratch version. Industrial frameworks (PyTorch, for example) sidestep the problem with a uniform tensor format: whether the "batch size" is 1 or 100, the rules for the output shape are always the same, so there's never any "sometimes a scalar, sometimes a list" ambiguity.

## 3. MLP — several layers in series

`MLP(nin, nouts)` chains `len(nouts)` `Layer`s end to end: layer 0 receives the raw input (dimension `nin`), and layer `k` receives the output of layer `k-1`. Each layer's `nin` is exactly the previous layer's `nout`:

```
Layer(nin,       nouts[0])   → outputs nouts[0] dimensions
Layer(nouts[0],  nouts[1])   → outputs nouts[1] dimensions
...
Layer(nouts[-2], nouts[-1])  → final output
```

The last layer usually switches off the nonlinearity (`nonlin=False`). `parameters()` walks every `Neuron` in every layer and flattens all the `w`s and `b`s into one flat list — during training, running `p.data -= lr * p.grad` on every `Value` in that list completes one step of gradient descent (GD).

### A closer look: why do we need a `parameters()` method?

An MLP has a nested structure: MLP → Layer → Neuron → Value. To reach every parameter you'd have to drill all the way down:

```python
# ❌ The awkward way
for layer in mlp.layers:
    for neuron in layer.neurons:
        for param in neuron.parameters():
            param.data -= lr * param.grad
```

Three nested loops! Imagine a 100-layer network — the code would be ugly.

**The solution: a `parameters()` method that flattens level by level**

```python
# ✅ The clean way
for param in mlp.parameters():
    param.data -= lr * param.grad
```

`mlp.parameters()` calls `layer.parameters()` internally, each `layer.parameters()` calls every `neuron.parameters()`, and in the end a **flat list** comes back:

```
mlp.parameters()
  → [all of Layer0's parameters, all of Layer1's parameters, ..., all of LayerN's parameters]
  → [every Neuron's w and b, flattened into one list]
```

**Why flatten?**

1. **Convenient for training**: one simple for loop visits all 41 parameters
2. **Optimizer-friendly**: optimizers such as SGD and Adam expect a list of parameters, not a nested structure
3. **Consistent with the frameworks**: PyTorch's `model.parameters()` likewise returns a generator that visits every parameter

The training loop in the next lesson, L58, uses this method.

### A closer look: why switch off the nonlinearity in the last layer?

This is a common source of confusion for beginners. The hidden layers all go through tanh (squashed to between -1 and 1), so why can't the last layer do the same?

**Scenario 1: a regression task (predicting house prices)**

Imagine you're predicting house prices. A price can be any positive number: $50,000, $500,000, $5,000,000...

- If the last layer goes through tanh, the output gets squashed into (-1, 1), say tanh(z) = 0.95
- How would you turn 0.95 back into a real house price? You can't! The information is lost
- The last layer **must be linear** (no activation function), so that the output z can be any real number

```python
# ❌ The wrong way: tanh on the last layer
pred = tanh(w[50]*h[50] + b)  # output squashed into (-1, 1), can't predict large prices

# ✅ The right way: a linear last layer
pred = w[50]*h[50] + b         # output can be any real number, e.g. 250000.5
```

**Scenario 2: a classification task (is it a cat?)**

Binary classification: output a probability (0 to 1) or a logit (any real number).

- The last layer outputs a logit (an unconstrained real number), which then goes through sigmoid or softmax to become a probability
- **The last layer is still linear**; the activation sits outside it

**Why do the hidden layers need a nonlinearity?**

Putting the hidden layers through an activation function (tanh/relu) serves a different purpose:
- it introduces a **nonlinear transformation**, so the network can learn complex relationships (not just linear combinations)
- it compresses the range of the data, stabilizing training

But **the last layer doesn't need this**; its job is to output the raw linear combination directly.

In [ ]:
# Demo: simulate the layer structure of MLP(3, [4, 4, 1]) with nested lists
# (print the structure first; the real class comes later)
layer_shapes = [(3, 4), (4, 4), (4, 1)]
total_params = sum(nin * nout + nout for nin, nout in layer_shapes)
print("MLP(3, [4, 4, 1]) parameter count:")
for i, (n_in, n_out) in enumerate(layer_shapes):
    w_count = n_in * n_out
    b_count = n_out
    print(f"  Layer {i}: {n_in}×{n_out} weights + {n_out} biases = {w_count + b_count}")
print(f"  total: {total_params} trainable parameters")

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        # ✏️ TODO: initialize self.w (nin Values, uniform random in [-1, 1)) and self.b (Value(0.0))
        # ✏️ TODO: store self.nonlin
        raise NotImplementedError("TODO: initialize self.w (nin random Values) and self.b (Value(0.0)), and store self.nonlin")

    def __call__(self, x):
        # ✏️ TODO: compute act = sum(wi * xi for ...) + self.b
        # ✏️ TODO: if self.nonlin is True, return act.tanh(); otherwise return act
        raise NotImplementedError("TODO: compute the weighted sum act; return act.tanh() if nonlin, otherwise act")

    def parameters(self):
        # ✏️ TODO: return self.w + [self.b]
        raise NotImplementedError("TODO: return the parameter list self.w + [self.b]")

    def __repr__(self):
        kind = "Tanh" if self.nonlin else "Linear"
        return f"{kind}Neuron({len(self.w)})"


In [ ]:
# ── Neuron unit check ──
try:
    import random; random.seed(0)
    n = Neuron(3)
    assert len(n.w) == 3, f'Neuron(3) should have 3 weights, got {len(n.w)}'
    assert isinstance(n.b, Value), 'b should be a Value'
    out = n([Value(1.0), Value(0.5), Value(-1.0)])
    assert isinstance(out, Value), '__call__ should return a Value'
    params = n.parameters()
    assert len(params) == 4, f'Neuron(3) should have 4 parameters, got {len(params)}'
    print('✅ Neuron passed: w×3 + b, __call__ returns a Value, parameters() length=4')
except (NotImplementedError, TypeError):
    print('⬜ Neuron not implemented yet')
except NameError:
    print('⬜ Run the cell that defines Value first (at the top of this lesson)')


In [ ]:
class Layer:
    def __init__(self, nin, nout, **kwargs):
        # ✏️ TODO: self.neurons = nout Neuron(nin, **kwargs)s
        raise NotImplementedError("TODO: create self.neurons, containing nout Neuron(nin, **kwargs)s")

    def __call__(self, x):
        # ✏️ TODO: outs = [n(x) for n in self.neurons]
        # ✏️ TODO: if len(outs)==1 return outs[0], otherwise return outs
        raise NotImplementedError("TODO: forward pass: build outs with a list comprehension; return a scalar when len==1, otherwise the list")

    def parameters(self):
        # ✏️ TODO: return the flattened list of every neuron's parameters()
        raise NotImplementedError("TODO: flatten all neuron.parameters() and return a single list")

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


In [ ]:
# ── Layer unit check ──
try:
    import random; random.seed(1)
    layer = Layer(2, 3)
    assert len(layer.neurons) == 3, f'Layer(2,3) should have 3 Neurons, got {len(layer.neurons)}'
    outs = layer([Value(0.5), Value(-0.5)])
    assert isinstance(outs, list) and len(outs) == 3, 'Layer(2,3) output should be a list of length 3'
    assert len(layer.parameters()) == 9, f'Layer(2,3) should have 9 parameters, got {len(layer.parameters())}'
    print('✅ Layer passed: 3 neurons, output list length=3, parameters() length=9')
    # a single-output layer should return a Value, not a list
    single = Layer(2, 1)([Value(0.5), Value(-0.5)])
    assert isinstance(single, Value), 'Layer(2,1) should return a single Value'
    print('✅ Layer(2,1) returns a Value (not a list) for a single output')
except (NotImplementedError, TypeError):
    print('⬜ Layer not implemented yet')
except NameError:
    print('⬜ Implement Neuron first')


### A closer look: why does MLP.__call__ "wrap everything into a list" at the end?

The `MLP.__call__` TODO you're about to implement contains a key comment:

> always return a list (if the result is not a list, `[x]`)

What does that mean, and why do it? First, an analogy.

**An analogy**: a delivery driver brings your order. Whether you bought 1 item or 3 this time, the driver always hands you a package (even if there's only 1 thing inside). When you receive it you never have to ask "is this one item pushed straight into my hands, or a box?" — you always open the same kind of package. If the driver sometimes pushed the item straight into your hands and sometimes boxed it, depending on their mood, you'd have to guess "which kind is it this time?" before every delivery — very easy to get wrong.

**Back to the MLP**: remember `Layer`'s "smart return" — a bare `Value` when `nout=1`, a list otherwise. For a regression task, an `MLP`'s last layer often has `nout=1` (predicting a single house-price number, say), and calling that layer on its own returns a bare `Value`, not a list.

If `MLP.__call__` does nothing and hands back the last layer's result as is, then:
- `MLP(3, [4, 4, 1])(x)` → returns a bare `Value`
- `MLP(3, [4, 4, 2])(x)` → returns a `[Value, Value]` list

**What trouble does that cause?** The training loop in the next lesson, `L58`, needs to pair "model outputs" with "training targets (labels)" one by one to compute the loss; the code looks roughly like this:

```python
# The core step of the L58 training loop (a preview)
for yp, yt in zip(ypred, ygt):     # assuming ypred and ygt are both lists
    loss += (yp - yt) ** 2
```

If `ypred` is sometimes a list and sometimes a bare `Value`, this `zip(ypred, ygt)` sometimes runs and sometimes fails (just like the `TypeError` we saw with our own eyes in the previous subsection). Whoever writes the training code would have to check "is it a Value or a list this time?" every single time — very easy to forget, and a bug waiting to happen.

**The fix**: have `MLP.__call__` do one final "uniform wrapping" **before returning to the outside caller** — whether the last layer gives a bare `Value` or a list, once it leaves `MLP.__call__` it is always a list (even with just 1 element). Then the `L58` training code can safely write `zip(ypred, ygt)` without handling "1 output" and "several outputs" as separate cases — which is why this step is required, not an optional nicety.

**How exactly is it implemented?** The logic is straightforward — check whether the result is a list, and if not, wrap it with `[x]`:

```python
def unify_output(raw_output):
    return raw_output if isinstance(raw_output, list) else [raw_output]
```

Run it and see what happens in each case, and how it spares next lesson's training code from type headaches:

In [ ]:
# Demo: the "wrap everything into a list" logic at the end of MLP.__call__

def unify_output(raw_output):
    # this is exactly the logic the "always return a list" TODO in MLP.__call__ asks for
    return raw_output if isinstance(raw_output, list) else [raw_output]

# Case 1: last layer nout=1 (a regression task), Layer returns a bare Value
single_val = Value(3.14)
print("Case 1 (regression, nout=1)")
print("  before wrapping:", single_val, " type:", type(single_val).__name__)
print("  after wrapping:", unify_output(single_val))

print()

# Case 2: last layer nout>=2 (a multi-output task), Layer returns a list
multi_val = [Value(1.0), Value(2.0)]
print("Case 2 (multi-output, nout=2)")
print("  before wrapping:", multi_val)
print("  after wrapping:", unify_output(multi_val))

print()

# Preview of the L58 training loop: with a uniform list, zip() is safe to use
ygt = [1.0]                          # 1 training sample, 1 target value
ypred = unify_output(single_val)     # after wrapping it is definitely a list, whatever it was before
loss = sum((yp - Value(yt))**2 for yp, yt in zip(ypred, ygt))
print(f"L58 preview: loss = {loss.data:.4f}  (no type worries for zip(ypred, ygt))")

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        # ✏️ TODO: sz = [nin] + nouts
        # ✏️ TODO: self.layers = a list of Layer(sz[i], sz[i+1], nonlin=...)
        #          the last layer nonlin=False, all others True
        raise NotImplementedError("TODO: build the sz list and create self.layers (last layer nonlin=False, all others True)")

    def __call__(self, x):
        # ✏️ TODO: feed x through each layer in turn (x = layer(x))
        # ✏️ TODO: always return a list (if the result is not a list, [x])
        raise NotImplementedError("TODO: chain the forward pass through every layer and return the final result wrapped as a list")

    def parameters(self):
        # ✏️ TODO: return a flat list of every parameter in every layer
        raise NotImplementedError("TODO: flatten all layer.parameters() and return a single list")

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


In [ ]:
# ── Combined MLP check (prints a hint instead of crashing when not implemented) ──
try:
    random.seed(0)

    # Basic structure check
    m = MLP(2, [3, 1])
    out = m([Value(0.5), Value(-0.5)])
    assert isinstance(out, list) and len(out) == 1, "MLP output should be a list of length 1"
    assert isinstance(out[0], Value), "MLP output elements should be Values"
    print("✅ MLP(2,[3,1]) forward pass passed, out[0].data =", round(out[0].data, 4))

    # Parameter count check
    m2 = MLP(3, [4, 4, 1])
    params = m2.parameters()
    assert len(params) == 41, f"there should be 41 parameters, got {len(params)}"
    print("✅ MLP(3,[4,4,1]) parameter count correct:", len(params))

    # Backprop check
    out2 = m2([Value(1.0), Value(2.0), Value(3.0)])
    loss = out2[0]
    loss.backward()
    grads = [p.grad for p in params]
    assert any(g != 0.0 for g in grads), "after backward() the parameter gradients should not all be 0"
    print("✅ backward() propagates gradients correctly, first parameter gradient =", round(params[0].grad, 6))
except (NotImplementedError, TypeError):
    print('⬜ TODO: finish the three classes Neuron / Layer / MLP first, then run this check cell.')

## Parameter experiment: verify the parameter total of MLP(3,[4,4,1])

Work out the parameter count of the three layers by hand, then check it with `len(m.parameters())`:

| Layer | nin | nout | Weights `nin × nout` | Biases (b) `nout` | Subtotal |
|----|-----|------|---------------------|---------------|------|
| 0  |  3  |  4   | 12                  | 4             | **16** |
| 1  |  4  |  4   | 16                  | 4             | **20** |
| 2  |  4  |  1   |  4                  | 1             | **5**  |
| Total | — | — | — | — | **41** |

What to expect: `len(m.parameters()) == 41`, and `Neuron.__repr__` shows `LinearNeuron` for the last layer (no nonlinearity).

In [ ]:
# ── Parameter total check (prints a hint instead of crashing when not implemented) ──
try:
    random.seed(99)
    m3 = MLP(3, [4, 4, 1])
    print("network structure:", m3)
    print()

    for i, layer in enumerate(m3.layers):
        lp = layer.parameters()
        print(f"Layer {i}: {len(lp):3d} parameters  |  neurons: {layer.neurons}")

    total = len(m3.parameters())
    print(f"\ntotal parameters: {total}  (expected 41)")

    # Check that the last layer has its nonlinearity switched off
    assert not m3.layers[-1].neurons[0].nonlin, "the last layer's Neurons should have nonlin=False (linear output)"
    print("✅ The last layer is linear (nonlin=False)")
except (NotImplementedError, TypeError):
    print('⬜ TODO: finish the three classes Neuron / Layer / MLP first, then run this cell to check the parameter count.')

## Wrap-up

This lesson assembled `Neuron` (a single weighted sum + activation), `Layer` (`nout` Neurons side by side) and `MLP` (several layers in series) into a complete network that can run forward, run backward and count its parameters; `parameters()` returns a flat list of all 41 trainable `Value`s, so gradient descent can sweep over every weight in one pass. This structure is exactly the same shape as PyTorch's `nn.Module`, and it is the prototype for the body of the network in the Aurora ML engine (`src/aurora/ml/`, a planned module). The next lesson, `L58_training_loop.ipynb`, uses this MLP to fit the two-moons dataset from `make_moons` (generated from scratch in NumPy if sklearn is missing), with hinge loss as the objective, tying together the full training loop "forward → compute loss → backward() → update parameters → zero the gradients"; the acceptance criterion is a classification accuracy above 85%.

## ✏️ Whiteboard challenge: work out the MLP structure by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: How many parameters does `MLP(2, [3, 1])` have in total? (List the weight + bias count of each layer.)
(Layer1: 3×(2+1)=9, Layer2: 1×(3+1)=4, 13 in total)

**Question 2**: What shape is the output of `Layer(4, 5)`'s `forward(x)`? What shape is the input x?
(Input: a list of 4 scalars; output: a list of 5 Values)

**Question 3**: Why does an `MLP`'s last layer usually have `nonlin=False`?
(The last layer is a regression output or a logit; it shouldn't be squashed or clipped by an activation such as tanh or relu)

**Question 4**: What does `Neuron.parameters()` return? Why do we need this method?
(It returns w + [b]; during training we need to visit every parameter to zero it with `p.grad = 0.0`)

**Question 5**: If `MLP(3, [4,4,1])` gets the input `[Value(1.0), Value(2.0), Value(3.0)]` (just like the combined check cell above), how many `Value` objects can you reach after the forward pass by walking back from the output along `_prev`? (Count the 3 inputs, the 41 parameters and every intermediate result.)
(Follow the reference implementation of `Neuron.__call__`: `sum((wi*xi ...), Value(0.0)) + self.b`, then `tanh()`. A Neuron with nin inputs owns nin+1 parameters, nin multiplications, 1 `Value(0.0)` seed for the sum, nin+1 additions, plus 1 tanh in a hidden layer: 3·nin+3 (+1) nodes. Layer 1: 4×13=52, layer 2: 4×16=64, layer 3: 1×15=15, plus the 3 inputs: 134 in all. The next cell works it out in detail.)

When you've finished the derivations, run the cell below to check.

## In depth: the number of nodes in the computation graph (a detailed derivation for whiteboard Question 5)

"How many nodes does the computation graph have?" looks like a simple question, but two things need pinning down first:

**What counts as a node?**

In a computation graph, **every `Value` object is a node**, whether it's a leaf you made yourself with `Value(1.0)` or the result of some operation. For example:

```python
a = Value(1.0)
b = Value(2.0)
c = a * b      # the multiplication creates a new node
d = c + a      # the addition creates a new node
```

This graph has 4 nodes (a, b, c, d), two of them created by operations. What Question 5 counts is: starting from the output node and walking back along `_prev`, how many `Value`s can you reach? That's exactly the pile of objects `backward()` has to topologically sort.

**How many nodes does one Neuron's forward pass really create?**

That depends on how `Neuron.__call__` is written. The reference implementation is:

```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
return act.tanh() if self.nonlin else act
```

Take `Neuron(3)`, with the input x being 3 `Value`s that already exist (they don't belong to this Neuron):

| Where it comes from | Count | Note |
|---|---|---|
| parameters w, b | 3 + 1 = 4 | leaves created in `__init__` |
| multiplications `wi * xi` | 3 | each weight multiplies its input once |
| `Value(0.0)` | 1 | the seed of `sum()`; the easiest one to miss |
| additions | 3 + 1 = 4 | `sum()` accumulates 3 times, then `+ self.b` once |
| `tanh()` | 1 | hidden layers have it, the output layer (`nonlin=False`) doesn't |
| **subtotal** | **13** (12 in the output layer) | |

In general, a Neuron with nin inputs owns **3·nin + 3** nodes, **+1** (tanh) in a hidden layer.

**The full count for MLP(3, [4,4,1])**

| Layer | Neurons | nin | Nodes per Neuron | Layer total |
|----|--------|-----|-------------|---------|
| L0 (tanh) | 4 | 3 | 3×3+3+1 = 13 | 4×13 = 52 |
| L1 (tanh) | 4 | 4 | 3×4+3+1 = 16 | 4×16 = 64 |
| L2 (linear) | 1 | 4 | 3×4+3 = 15 | 1×15 = 15 |
| inputs | | | 3 `Value`s | 3 |
| **Total** | | | | **52+64+15+3 = 134** |

Cross-check from another angle: 41 parameters + 32 multiplications (12+16+4) + 41 additions (16+20+5) + 9 `Value(0.0)`s + 8 tanh + 3 inputs = 134. It adds up. The answer-check cell below really walks the graph along `_prev` to count.

**A small trap**: if you pass the input as plain floats, `[1.0, 2.0, 3.0]`, then `__mul__` inside `wi * xi` wraps a fresh `Value(xi)` for every single multiplication: the 4 Neurons of the first layer wrap 3 each, the input nodes go from 3 to 12, and the total becomes 143. Same numbers, different graph. That's why this lesson's check cells wrap the inputs in `Value` before feeding them in.

The point is: **the number of nodes in the computation graph grows quickly with nin (the input dimension) and the width of the network**: a tiny network with only 41 parameters has to store 134 objects for a single forward pass. That's why backward(), although it gets the gradients right, becomes expensive for large networks.

(What deep-learning frameworks do here is fuse operations into big tensor-level ops and add computation-graph optimization, which greatly cuts the number of nodes to store and walk.)


In [ ]:
# A real count: how many nodes does the computation graph have?

def count_graph_nodes(root):
    """Walk back from root along _prev and count how many Values you can reach (root included)."""
    seen, stack = set(), [root]
    while stack:
        v = stack.pop()
        if id(v) in seen:
            continue
        seen.add(id(v))
        stack.extend(v._prev)
    return len(seen)

# Unroll one Neuron(3) by hand, written the way the reference Neuron.__call__ does it
w = [Value(0.5), Value(-0.3), Value(0.2)]
x = [Value(1.0), Value(2.0), Value(3.0)]
b = Value(0.1)

act = sum((wi * xi for wi, xi in zip(w, x)), Value(0.0)) + b   # 3 multiplications + 1 Value(0.0) + 4 additions
y = act.tanh()                                                 # 1 tanh

print(f"output of Neuron(3): y = {y.data:.4f}")
print(f"walking back from y along _prev reaches {count_graph_nodes(y)} nodes:")
print("  3 inputs + 4 parameters + 3 multiplications + 1 Value(0.0) + 4 additions + 1 tanh = 16")

# The general formula: a Neuron owns 3*nin+3 nodes of its own (inputs not counted), +1 tanh in a hidden layer
print("\nThe general pattern (inputs not counted):")
for nin in [2, 3, 4, 5]:
    print(f"  Neuron({nin}): {3 * nin + 4} in a hidden layer, {3 * nin + 3} in the output layer")


In [ ]:
# ✏️ Answer-check cell
import random

# Q1: MLP(2,[3,1]) parameter count = 13
try:
    random.seed(0)
    m_check = MLP(2, [3, 1])
    params = m_check.parameters()
    assert len(params) == 13, f"parameter count={len(params)}, expected 13"
    print(f"Q1 ✅  MLP(2,[3,1]) parameter count={len(params)} (Layer1:9 + Layer2:4 = 13)")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  MLP(2,[3,1]) parameter count: Layer1=3×(2+1)=9, Layer2=1×(3+1)=4, 13 in total (to be verified once implemented)")

# Q2: Layer(4,5) outputs 5 Values
try:
    random.seed(1)
    layer_check = Layer(4, 5)
    x_in = [Value(float(i)) for i in range(4)]
    out_check = layer_check(x_in)
    assert len(out_check) == 5, f"output length={len(out_check)}"
    print(f"Q2 ✅  Layer(4,5): 4 input Values → {len(out_check)} output Values")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  Layer(4,5): input list of length 4, output list of length 5 (to be verified once implemented)")

# Q3: last layer nonlin=False (concept check)
try:
    random.seed(2)
    mlp_check = MLP(2, [3, 1])
    last_layer = mlp_check.layers[-1]
    assert all(not n.nonlin for n in last_layer.neurons), "the last layer should have nonlin=False"
    print(f"Q3 ✅  every Neuron in the MLP's last layer has nonlin=False (the regression output isn't clipped)")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  last layer nonlin=False: a regression output / logit shouldn't be clipped by an activation (to be verified once implemented)")

# Q4: length of Neuron.parameters()
try:
    random.seed(3)
    n_check = Neuron(5)
    assert len(n_check.parameters()) == 6, f"parameter count={len(n_check.parameters())}"
    print(f"Q4 ✅  Neuron(5).parameters() has {len(n_check.parameters())} entries (5 w's + 1 b)")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  Neuron(nin).parameters() = [w0,...,w_(nin-1), b], nin+1 in total (to be verified once implemented)")

# Q5: how many Value nodes the computation graph holds after one forward pass of MLP(3,[4,4,1])
def _count_graph_nodes(root):
    seen, stack = set(), [root]
    while stack:
        v = stack.pop()
        if id(v) in seen:
            continue
        seen.add(id(v))
        stack.extend(v._prev)
    return len(seen)

try:
    random.seed(4)
    m3_check = MLP(3, [4, 4, 1])
    n_params = len(m3_check.parameters())
    # parameters: L1: 4×(3+1)=16, L2: 4×(4+1)=20, L3: 1×(4+1)=5 → 41
    assert n_params == 41, f"parameter count={n_params}"
    out5 = m3_check([Value(1.0), Value(2.0), Value(3.0)])[0]
    n_nodes = _count_graph_nodes(out5)
    # nodes: L0: 4×13=52, L1: 4×16=64, L2: 1×15=15, plus 3 inputs → 134
    assert n_nodes == 134, (f"node count={n_nodes}, expected 134. If your Neuron.__call__ is written differently from the reference"
                            f" (e.g. without sum(..., Value(0.0))), redo the derivation for your own version")
    print(f"Q5 ✅  MLP(3,[4,4,1]) one forward pass: {n_params} parameters, {n_nodes} Value nodes in the graph (L0:52 + L1:64 + L2:15 + inputs:3 = 134)")
except (NotImplementedError, TypeError):
    print(f"Q5 ⬜  the node count of MLP(3,[4,4,1]) can only be measured once Neuron / Layer / MLP are implemented: expected L0:52 + L1:64 + L2:15 + inputs:3 = 134")
print("\n🎉 MLP structure whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l57_review = {
    "neuron_class_impl":       None,  # Neuron.__init__/__call__/parameters implemented correctly? True/False
    "layer_class_impl":        None,  # Layer.__init__/__call__/parameters implemented correctly? True/False
    "mlp_class_impl":          None,  # MLP.__init__/__call__/parameters implemented correctly? True/False
    "param_count_understood":  None,  # can you count the parameters of any MLP structure by hand? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l57_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l57_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L57 fully cleared! On to L58: the training loop')

---

→ **Next lesson** · [L58 · The training loop](L58_training_loop.ipynb)

> Next lesson covers **the training loop**: computing the loss, `optimizer.step`, convergence curves, and fitting the `make_moons` dataset.